In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


Mounted at /content/gdrive


In [ ]:
!apt-get update -qq && apt-get install -y ffmpeg build-essential
!pip install -q librosa soundfile numpy scipy pandas tqdm pyworld
print("✓ Dependencies installed")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
✓ Dependencies installed


In [ ]:
import os

LANGUAGES = {
    'odia': {
        'input': '/content/drive/MyDrive/five_songs_dataset/odia',
        'output': '/content/drive/MyDrive/five_language_output/odia/rvc'
    },
    'punjabi': {
        'input': '/content/drive/MyDrive/five_songs_dataset/punjabi_folk_songs',
        'output': '/content/drive/MyDrive/five_language_output/punjabi/rvc'
    },
    'tamil': {
        'input': '/content/drive/MyDrive/five_songs_dataset/tamil',
        'output': '/content/drive/MyDrive/five_language_output/tamil/rvc'
    },
    'telugu': {
        'input': '/content/drive/MyDrive/five_songs_dataset/telugu',
        'output': '/content/drive/MyDrive/five_language_output/telugu/rvc'
    }
}

for lang in LANGUAGES.values():
    os.makedirs(lang['output'], exist_ok=True)
    os.makedirs(f"{lang['output']}/references", exist_ok=True)

print("✓ Output directories created")


✓ Output directories created


In [ ]:
def load_audio_files(input_dir):
    audio_files = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(('.mp3', '.wav', '.flac', '.m4a')):
                audio_files.append(os.path.join(root, file))
    return sorted(audio_files)

dataset_info = {}
for lang, config in LANGUAGES.items():
    files = load_audio_files(config['input'])
    dataset_info[lang] = files
    print(f"{lang.capitalize()}: {len(files)} songs found")

total_songs = sum(len(f) for f in dataset_info.values())
print(f"✓ Total songs across all languages: {total_songs}")


Odia: 15 songs found
Punjabi: 71 songs found
Tamil: 10 songs found
Telugu: 50 songs found
✓ Total songs across all languages: 146


In [ ]:
import librosa
import soundfile as sf

def create_reference_sample(audio_path, output_path, duration=20):
    try:
        audio, sr = librosa.load(audio_path, sr=44100)
        total_samples = len(audio)
        start_pos = total_samples // 3
        sample_length = sr * duration
        sample = audio[start_pos:start_pos + sample_length] if start_pos + sample_length <= total_samples else audio[-sample_length:] if total_samples >= sample_length else audio
        sample = librosa.util.normalize(sample)
        sf.write(output_path, sample, 44100)
        return True
    except Exception as e:
        print(f"Error creating reference: {e}")
        return False

reference_samples = {}

for lang, files in dataset_info.items():
    if files:
        ref_path = f"{LANGUAGES[lang]['output']}/references/{lang}_rvc_reference.wav"
        ok = create_reference_sample(files[0], ref_path)
        if ok:
            reference_samples[lang] = ref_path
            print(f"{lang.capitalize()}: Reference created from {os.path.basename(files[0])}")

print("✓ Reference samples for RVC created")


Error creating reference: [Errno 2] No such file or directory: '/content/drive/MyDrive/five_songs_dataset/odia/Alo Bau Lo Bau _ Folk _ Sudha Mishra.mp3'
Error creating reference: [Errno 2] No such file or directory: '/content/drive/MyDrive/five_songs_dataset/punjabi_folk_songs/3 Peg.mp3'
Error creating reference: [Errno 2] No such file or directory: '/content/drive/MyDrive/five_songs_dataset/tamil/004_Hello Brother.mp3.mp3'
Error creating reference: [Errno 2] No such file or directory: '/content/drive/MyDrive/five_songs_dataset/telugu/001_Lachimi Na Chinni Lachimi.mp3.mp3'
✓ Reference samples for RVC created


/tmp/ipython-input-2931970912.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(audio_path, sr=44100)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [ ]:
# Install dependencies
!apt-get update -qq && apt-get install -y ffmpeg build-essential
!pip install -q librosa soundfile numpy scipy tqdm pyworld

import os
import librosa
import soundfile as sf
import numpy as np
import pyworld as pw
from tqdm import tqdm
import gc

# Language dataset and output paths (excluding Bengali)
LANGUAGES = {
    'odia': {
        'input': '/content/drive/MyDrive/five_songs_dataset/odia',
        'output': '/content/drive/MyDrive/five_language_output/odia/rvc'
    },
    'punjabi': {
        'input': '/content/drive/MyDrive/five_songs_dataset/punjabi_folk_songs',
        'output': '/content/drive/MyDrive/five_language_output/punjabi/rvc'
    },
    'tamil': {
        'input': '/content/drive/MyDrive/five_songs_dataset/tamil',
        'output': '/content/drive/MyDrive/five_language_output/tamil/rvc'
    },
    'telugu': {
        'input': '/content/drive/MyDrive/five_songs_dataset/telugu',
        'output': '/content/drive/MyDrive/five_language_output/telugu/rvc'
    }
}

# Create output directories
for lang, conf in LANGUAGES.items():
    os.makedirs(conf['output'], exist_ok=True)
    os.makedirs(f"{conf['output']}/references", exist_ok=True)

def load_audio_files(path):
    return [os.path.join(root, f) for root, _, files in os.walk(path)
           for f in files if f.lower().endswith(('.wav', '.mp3', '.flac', '.m4a'))]

def create_reference_sample(audio_path, ref_path, duration=30):
    audio, sr = librosa.load(audio_path, sr=22050, duration=duration)
    sample = librosa.util.normalize(audio)
    sf.write(ref_path, sample, sr)

def rvc_convert(source, reference, sr):
    source = source.astype(np.float64)
    reference = reference.astype(np.float64)
    f0_src, t_src = pw.dio(source, sr)
    f0_src = pw.stonemask(source, f0_src, t_src, sr)
    sp_src = pw.cheaptrick(source, f0_src, t_src, sr)
    ap_src = pw.d4c(source, f0_src, t_src, sr)
    f0_ref, t_ref = pw.dio(reference, sr)
    f0_ref = pw.stonemask(reference, f0_ref, t_ref, sr)
    pitch_ratio = np.mean(f0_ref[f0_ref>0]) / np.mean(f0_src[f0_src>0]) if np.mean(f0_src[f0_src>0]) > 0 else 1.0
    f0_converted = np.clip(f0_src * pitch_ratio, 50, 800)
    converted = pw.synthesize(f0_converted, sp_src, ap_src, sr)
    converted = librosa.util.normalize(converted)
    converted, _ = librosa.effects.trim(converted, top_db=20)
    return converted

# Main Processing Loop without Bengali
for lang, conf in LANGUAGES.items():
    print(f"\n=== Processing {lang.upper()} ===")
    files = load_audio_files(conf['input'])
    if not files:
        print(f"No files found for {lang}, skipping.")
        continue

    ref_path = f"{conf['output']}/references/{lang}_rvc_ref.wav"
    create_reference_sample(files[0], ref_path)
    reference, sr = librosa.load(ref_path, sr=22050)

    for i, audio_path in enumerate(tqdm(files[1:], desc=f"RVC {lang}")):
        out_file = f"rvc_{lang}_{i+1:03d}.wav"
        out_path = os.path.join(conf['output'], out_file)
        try:
            source, _ = librosa.load(audio_path, sr=22050, duration=30)
            converted_audio = rvc_convert(source, reference, sr)
            sf.write(out_path, converted_audio, sr, subtype='PCM_16')
            print(f"✓ Saved {out_file}")
        except Exception as e:
            print(f"Error on {out_file}: {e}")
        del source, converted_audio
        gc.collect()
    del reference
    gc.collect()

print("\nProcessing complete (excluding [translate:বাংলা]). Outputs saved in respective folders.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 21.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

=== Processing ODIA ===


RVC odia:   7%|▋         | 1/14 [00:09<01:57,  9.06s/it]

✓ Saved rvc_odia_001.wav


RVC odia:  14%|█▍        | 2/14 [00:14<01:21,  6.77s/it]

✓ Saved rvc_odia_002.wav


...

✓ Saved rvc_telugu_047.wav


RVC telugu:  98%|█████████▊| 48/49 [04:16<00:04,  4.92s/it]

✓ Saved rvc_telugu_048.wav


RVC telugu: 100%|██████████| 49/49 [04:23<00:00,  5.37s/it]

✓ Saved rvc_telugu_049.wav

Processing complete (excluding [translate:বাংলা]). Outputs saved in respective folders.
